# MLFlowTransformer によるバッチ推論

In [ ]:
from synapse.ml.predict import MLFlowTransformer

df_test = spark.read.format("delta").load("Tables/dbo/walmart_sales_test")

model = MLFlowTransformer(
    inputCols=["Store", "Temperature", "Fuel_Price", "CPI", "Unemployment"],
    outputCol='predictions',
    modelName='nb_automl-AutoMLModel',
    modelVersion=1
)

predictions = model.transform(df_test)
display(predictions)

In [ ]:
predictions.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("pred_MLFlowTransformers")

# PREDICT句によるバッチ推論

In [ ]:
from pyspark.ml.feature import SQLTransformer 

df_test = spark.read.format("delta").load("Tables/dbo/walmart_sales_test")

model_name = 'nb_automl-AutoMLModel'
model_version = 1
features = ["Store", "Temperature", "Fuel_Price", "CPI", "Unemployment"]

sqlt = SQLTransformer().setStatement( 
    f"SELECT PREDICT('{model_name}/{model_version}', {','.join(features)}) as predictions FROM __THIS__")

display(sqlt.transform(df_test))

# PySpark UDFによるバッチ推論

In [ ]:
from pyspark.sql.functions import col, pandas_udf, udf, lit
from synapse.ml.predict import MLFlowTransformer

df_test = spark.read.format("delta").load("Tables/dbo/walmart_sales_test")
df_test = df_test.drop("Date")
df_test = df_test.drop("Weekly_Sales")
df_test = df_test.drop("Holiday_Flag")

model = MLFlowTransformer(
    inputCols=["Store", "Temperature", "Fuel_Price", "CPI", "Unemployment"],
    outputCol='predictions',
    modelName='nb_automl-AutoMLModel',
    modelVersion=1
)

my_udf = model.to_udf()
features = df_test.columns

display(df_test.withColumn("predictions", my_udf(*[col(f) for f in features])))